# 参数量、激活、FLOPs、显存的手算公式

> 前面几节我们建过 MiniGPT、读过 SmolLM2 的 config.json、估过 7B 模型在 8 卡下的 ZeRO 显存。所有这些数字背后其实共享一套底层公式：参数怎么数、激活怎么数、FLOPs 怎么数、显存怎么数。把这套公式吃透，任何模型拿到 config 就能秒算它的训练/推理开销。
>
> 这一节把这件事系统做一遍。先逐项推导参数量、激活、FLOPs 的闭式公式，再用 PyTorch 实测和公式对拍；然后展开推理显存和训练显存两套账本，覆盖 KV cache、AdamW 状态、gradient checkpointing；最后专门讨论 MoE 场景下哪些公式要改写。

LLM 工程里几乎每一道面试题、每一次容量规划，最后都要落到同一个问题：给定 hidden size $D$、层数 $L$、词表 $V$、batch $B$、序列长 $S$，这个模型一次 forward 多少 FLOPs？训练时显存占多少？推理时 KV cache 多大？

这些问题有一个共同特点：都能用纸笔算出来。公式不复杂，关键是把每个矩阵的形状和它对应的乘加次数对应清楚。本文采用一套统一的符号记号：

- $D$ = hidden size，Block 内部统一宽度
- $L$ = 层数
- $V$ = 词表大小
- $H$ = 注意力头数，$K$ = KV 头数（GQA 下 $K < H$），$d_h = D / H$ 为每个头的维度
- $F$ = FFN 中间层宽度（SwiGLU 配置下通常 $F \approx 8D/3$）
- $B$ = batch size，$S$ = 序列长度
- $P$ = 模型总参数量

下面每个 section 都遵循同一个套路：先用这套符号推导闭式公式，再用 `sum(p.numel())` 或 `torch.cuda.memory_allocated()` 实测，最后做「预测 vs 实测」的对照。

## 1. 参数量：从单层到全局

先看一个 Transformer Block 里有哪些可训练权重。Llama-style 的 Block 由三部分构成：Attention（四个投影 Q/K/V/O）、FFN（三个投影 gate/up/down，对应 SwiGLU）、两个 RMSNorm。把每个权重的形状写出来，参数量就清楚了。

In [1]:
import torch
import torch.nn as nn

# 用 SmolLM2-135M 的真实配置做演示
D = 576      # hidden size
H = 9        # num_attention_heads
K = 3        # num_key_value_heads (GQA)
L = 30       # num_hidden_layers
V = 49152    # vocab_size
FF = 1536    # intermediate_size (FFN)
d_h = D // H  # head_dim = 64

# --- Attention 四个投影 ---
# 标准 MHA: Q,K,V 都是 [D, H*d_h]
# GQA:     K,V 是 [D, K*d_h]，只有 Q 是 [D, H*d_h]
q_params = D * (H * d_h)           # = D * D
k_params = D * (K * d_h)           # GQA 下小一些
v_params = D * (K * d_h)
o_params = (H * d_h) * D           # = D * D

attn_params = q_params + k_params + v_params + o_params
print("=== 单层 Attention 参数（GQA） ===")
print(f"Q: [{D}, {H*d_h}]  → {q_params:,}")
print(f"K: [{D}, {K*d_h}]  → {k_params:,}  (GQA 缩小为 Q 的 {K/H:.2f})")
print(f"V: [{D}, {K*d_h}]  → {v_params:,}")
print(f"O: [{H*d_h}, {D}]  → {o_params:,}")
print(f"合计: {attn_params:,}")

# 在标准 MHA（K=H）下，4 个投影加起来约等于 4D^2
mha_attn = 4 * D * D
print(f"\n若改回 MHA（K=H={H}），Attention 参数 ≈ 4D^2 = {mha_attn:,}")
print(f"  公式 4D^2 在 D={D} 时给出 {4*D*D:,}，与 MHA 精确值相同。")

=== 单层 Attention 参数（GQA） ===
Q: [576, 576]  → 331,776
K: [576, 192]  → 110,592  (GQA 缩小为 Q 的 0.33)
V: [576, 192]  → 110,592
O: [576, 576]  → 331,776
合计: 884,736

若改回 MHA（K=H=9），Attention 参数 ≈ 4D^2 = 1,327,104
  公式 4D^2 在 D=576 时给出 1,327,104，与 MHA 精确值相同。


Attention 在 MHA 下的简化形式是 $4D^2$。接下来看 FFN。SwiGLU FFN 有三个矩阵：gate、up、down，每个都涉及 $D$ 和 $F$ 一次乘法。

| 投影 | 形状 | 参数量 |
|:---|:---|:---|
| gate | $[D, F]$ | $DF$ |
| up   | $[D, F]$ | $DF$ |
| down | $[F, D]$ | $DF$ |

三项相加得到 $3DF$。这里有一个工程上的考量：现代模型的 $F$ 普遍选 $8D/3$，原因是把 FFN 总参数量控制在「Attention 的 $4D^2$ 两倍」附近（$3D \cdot \tfrac{8D}{3} = 8D^2 \approx 2 \times 4D^2$），同时为了让 $F$ 能被 64 整除，常常向上 round 一下。

In [2]:
# --- FFN 三个投影（SwiGLU） ---
gate_p = D * FF
up_p   = D * FF
down_p = FF * D
ffn_params = gate_p + up_p + down_p

print("=== 单层 FFN 参数（SwiGLU） ===")
print(f"gate: [{D}, {FF}]  → {gate_p:,}")
print(f"up:   [{D}, {FF}]  → {up_p:,}")
print(f"down: [{FF}, {D}]  → {down_p:,}")
print(f"合计: {ffn_params:,}  (公式 3DF = 3·{D}·{FF} = {3*D*FF:,})")

# 8D/3 的来源：让 FFN 参数 ≈ 2 倍 Attention 参数
F_ideal = 8 * D / 3
print(f"\nF 的「理想值」= 8D/3 = {F_ideal:.1f}")
print(f"实际配置 F = {FF}（向上 round 到 64 的倍数: {((int(F_ideal) + 63) // 64) * 64}）")
print(f"代回 FFN 参数: 3D·(8D/3) = 8D² = {8*D*D:,}  ≈ Attention {4*D*D:,} 的两倍")

=== 单层 FFN 参数（SwiGLU） ===
gate: [576, 1536]  → 884,736
up:   [576, 1536]  → 884,736
down: [1536, 576]  → 884,736
合计: 2,654,208  (公式 3DF = 3·576·1536 = 2,654,208)

F 的「理想值」= 8D/3 = 1536.0
实际配置 F = 1536（向上 round 到 64 的倍数: 1536）
代回 FFN 参数: 3D·(8D/3) = 8D² = 2,654,208  ≈ Attention 1,327,104 的两倍


**单层小计和全局总参数**

每层 RMSNorm 只有 $D$ 个可训练的缩放参数（没有 bias），两个 RMSNorm 加起来 $2D$。所以单层 Block 的参数量是：

$$P_{\text{layer}} = \underbrace{4D^2}_{\text{Attention}} + \underbrace{3DF}_{\text{FFN}} + \underbrace{2D}_{\text{Norm}} \approx 12D^2 \;\;\text{(代入 } F=8D/3 \text{)}$$

全局加上首尾的 embedding（$V \cdot D$）和 unembedding（$V \cdot D$），如果不 tie 就是 $2VD$；tie 的话只剩 $VD$。常用估算公式：

$$P \approx 2VD + 12LD^2$$

In [3]:
# --- 单层和全局公式 ---
norm_params = 2 * D
per_layer = attn_params + ffn_params + norm_params
print(f"=== 单层小计 ===")
print(f"Attention: {attn_params:,}  (公式 4D² ≈ {4*D*D:,})")
print(f"FFN:       {ffn_params:,}  (公式 3DF = {3*D*FF:,})")
print(f"Norm:      {norm_params:,}")
print(f"单层合计:  {per_layer:,}  (公式 12D² ≈ {12*D*D:,}, 误差 {abs(per_layer-12*D*D)/per_layer*100:.2f}%)")

# --- 全局总参数量公式 ---
total_formula = 2 * V * D + L * 12 * D * D   # 估算
total_exact = V * D + L * per_layer + D       # tie word embeddings 的精确值

print(f"\n=== 全局总参数 ===")
print(f"Embedding (V·D):        {V*D:>12,}")
print(f"{L} 层 Block (L·单层):    {L*per_layer:>12,}")
print(f"final Norm:             {D:>12,}")
print(f"精确合计 (tie):         {total_exact:>12,}  ≈ {total_exact/1e6:.1f}M")
print(f"估算公式 2VD+12LD²:     {total_formula:>12,}  (偏高，因未考虑 tie 和 GQA)")

=== 单层小计 ===
Attention: 884,736  (公式 4D² ≈ 1,327,104)
FFN:       2,654,208  (公式 3DF = 2,654,208)
Norm:      1,152
单层合计:  3,540,096  (公式 12D² ≈ 3,981,312, 误差 12.46%)

=== 全局总参数 ===
Embedding (V·D):          28,311,552
30 层 Block (L·单层):     106,202,880
final Norm:                      576
精确合计 (tie):          134,515,008  ≈ 134.5M
估算公式 2VD+12LD²:      176,062,464  (偏高，因未考虑 tie 和 GQA)


In [4]:
# --- 用 PyTorch 建出完整模型，sum(p.numel()) 实测对拍 ---
import torch.nn.functional as F

class TransformerBlock(nn.Module):
    """Llama-style Block: RMSNorm → GQA Attention → RMSNorm → SwiGLU FFN"""
    def __init__(self, d, h, kv, d_head, ff_dim):
        super().__init__()
        self.attn_norm = nn.RMSNorm(d)
        self.ffn_norm  = nn.RMSNorm(d)
        self.q_proj = nn.Linear(d, h * d_head, bias=False)
        self.k_proj = nn.Linear(d, kv * d_head, bias=False)
        self.v_proj = nn.Linear(d, kv * d_head, bias=False)
        self.o_proj = nn.Linear(h * d_head, d, bias=False)
        self.gate = nn.Linear(d, ff_dim, bias=False)
        self.up   = nn.Linear(d, ff_dim, bias=False)
        self.down = nn.Linear(ff_dim, d, bias=False)
        self.h, self.kv, self.d_head = h, kv, d_head

    def forward(self, x):
        # 简化实现：不包含 RoPE 和 causal mask
        residual = x
        x = self.attn_norm(x)
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.d_head).transpose(1, 2)
        groups = self.h // self.kv
        k = k.repeat_interleave(groups, dim=1)
        v = v.repeat_interleave(groups, dim=1)
        scale = self.d_head ** -0.5
        attn = (q @ k.transpose(-2, -1)) * scale
        attn = F.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, -1)
        x = residual + self.o_proj(out)
        residual = x
        x = self.ffn_norm(x)
        x = residual + self.down(F.silu(self.gate(x)) * self.up(x))
        return x

class MiniLM(nn.Module):
    def __init__(self, V, D, L, H, K, FF):
        super().__init__()
        d_h = D // H
        self.embed = nn.Embedding(V, D)
        self.blocks = nn.ModuleList(
            [TransformerBlock(D, H, K, d_h, FF) for _ in range(L)])
        self.final_norm = nn.RMSNorm(D)
    def forward(self, idx):
        x = self.embed(idx)
        for blk in self.blocks: x = blk(x)
        x = self.final_norm(x)
        return F.linear(x, self.embed.weight)   # tie

torch.manual_seed(42)
model = MiniLM(V, D, L, H, K, FF)
total_real = sum(p.numel() for p in model.parameters())

print("=== 公式 vs PyTorch 实测 ===")
print(f"公式预测: {total_exact:,}")
print(f"PyTorch:  {total_real:,}")
print(f"误差:     {abs(total_exact - total_real):,}  ({'完全对上' if total_exact==total_real else '需检查'})")

=== 公式 vs PyTorch 实测 ===
公式预测: 134,515,008
PyTorch:  134,515,008
误差:     0  (完全对上)


## 2. 激活内存：每一层 forward 临时存了什么

参数量关心「权重一共多少」，激活关心「前向时中间结果占多少显存」。激活和 batch、序列长度强相关，而参数量和它们无关——这是两者最大的区别。

把单层 Block 在 forward 时产生的所有张量列出来，逐项求和：

**Attention 部分的激活**（MHA，无 FlashAttention）

| 张量 | 形状 | 大小 |
|:---|:---|:---|
| RMSNorm 输入 | $[B, S, D]$ | $BSD$ |
| RMSNorm 输出 | $[B, S, D]$ | $BSD$ |
| Q, K, V | $[B, S, D]$ × 3 | $3BSD$ |
| Attention scores $QK^\top$ | $[B, H, S, S]$ | $BHS^2 = BS^2 \cdot (H)$ |
| Attention 输出 | $[B, S, D]$ | $BSD$ |

把上面这些加起来约 $6BSD + BHS^2$（其中 $BHS^2 = BS^2 \cdot H = BNS^2$，这里 $N=H$ 是头数）。第二项 $BNS^2$ 是 $S \times S$ attention matrix 的总开销。

In [5]:
# --- 演示 Attention 各张量的实际形状 ---
B, S = 2, 16
x = torch.randn(B, S, D)

# 模拟 Attention 前向，记录每个中间张量的字节数（FP32 = 4 bytes）
def show(name, t):
    print(f"  {name:<22} shape={list(t.shape)}  bytes={t.nelement()*4:,}")

print(f"=== Attention 激活清单（B={B}, S={S}, D={D}, H={H}） ===")
norm_in = x                                         # RMSNorm 输入
show("norm_in", norm_in)
norm_out = norm_in                                  # RMSNorm 输出
show("norm_out", norm_out)
q = torch.randn(B, S, H * d_h)
k = torch.randn(B, S, K * d_h)
v = torch.randn(B, S, K * d_h)
show("Q/K/V 合计", torch.cat([q, k, v], dim=-1))
scores = torch.randn(B, H, S, S)
show("attn_scores QK^T", scores)
attn_out = torch.randn(B, S, D)
show("attn_out", attn_out)

# 公式校验
attn_act_formula = 6 * B * S * D + B * H * S * S
attn_act_bytes = (B*S*D + B*S*D + B*S*(H+2*K)*d_h + B*H*S*S + B*S*D) * 4
print(f"\n公式 6BSD + BHS² = {attn_act_formula:,} elements")
print(f"  = {attn_act_formula*4/1024:.1f} KB（FP32）")

=== Attention 激活清单（B=2, S=16, D=576, H=9） ===
  norm_in                shape=[2, 16, 576]  bytes=73,728
  norm_out               shape=[2, 16, 576]  bytes=73,728
  Q/K/V 合计               shape=[2, 16, 960]  bytes=122,880
  attn_scores QK^T       shape=[2, 9, 16, 16]  bytes=18,432
  attn_out               shape=[2, 16, 576]  bytes=73,728

公式 6BSD + BHS² = 115,200 elements
  = 450.0 KB（FP32）


**FFN 部分的激活**

| 张量 | 形状 | 大小 |
|:---|:---|:---|
| RMSNorm 输入 | $[B, S, D]$ | $BSD$ |
| gate 输出 | $[B, S, F]$ | $BSF$ |
| up 输出 | $[B, S, F]$ | $BSF$ |
| down 输出 | $[B, S, D]$ | $BSD$ |

加起来约 $2BSD + 2BSF$，代入 $F = 8D/3$ 得到 $2BSD + 2BS \cdot \tfrac{8D}{3} = 2BSD + \tfrac{16}{3}BSD \approx 6.33BSD$，常简化为 $8BSD$。

**单层激活合计**：Attention 的 $6BSD + BNS^2$ 加 FFN 的 $8BSD$，得到经典的 $14BSD + BNS^2$。

In [6]:
# --- 单层激活公式 14BSD + BNS² ---
B, S = 8, 4096
per_layer_act = 14 * B * S * D + B * H * S * S

print(f"=== 单层激活公式（B={B}, S={S}, D={D}, H={H}） ===")
print(f"Attention: 6BSD + BHS² = {6*B*S*D + B*H*S*S:,} elements")
print(f"FFN:       8BSD       = {8*B*S*D:,} elements")
print(f"单层合计:  14BSD+BHS² = {per_layer_act:,} elements")
print(f"  = {per_layer_act * 4 / 1e9:.2f} GB（FP32）")
print(f"  = {per_layer_act * 2 / 1e9:.2f} GB（BF16）")

# 全 L 层
total_act = L * per_layer_act
print(f"\n{L} 层合计: {total_act:,} elements")
print(f"  = {total_act * 4 / 1e9:.2f} GB（FP32）")
print(f"  = {total_act * 2 / 1e9:.2f} GB（BF16）")

# 看哪一项主导
linear_part = L * 14 * B * S * D
quad_part = L * B * H * S * S
print(f"\n=== 主导项分析 ===")
print(f"线性项 14·BSD·L = {linear_part*4/1e9:.2f} GB ({linear_part/per_layer_act/L*100:.1f}% of per-layer)")
print(f"平方项 BHS²·L   = {quad_part*4/1e9:.2f} GB ({quad_part/per_layer_act/L*100:.1f}% of per-layer)")
print(f"  S={S} 时，S² 项已经接近线性项。")

=== 单层激活公式（B=8, S=4096, D=576, H=9） ===
Attention: 6BSD + BHS² = 1,321,205,760 elements
FFN:       8BSD       = 150,994,944 elements
单层合计:  14BSD+BHS² = 1,472,200,704 elements
  = 5.89 GB（FP32）
  = 2.94 GB（BF16）

30 层合计: 44,166,021,120 elements
  = 176.66 GB（FP32）
  = 88.33 GB（BF16）

=== 主导项分析 ===
线性项 14·BSD·L = 31.71 GB (17.9% of per-layer)
平方项 BHS²·L   = 144.96 GB (82.1% of per-layer)
  S=4096 时，S² 项已经接近线性项。


## 3. FLOPs：一次 forward 要做多少次乘加

FLOPs（floating point operations）是衡量算力的基本单位。一次乘法和一次加法各算 1 op，所以一个矩阵乘 $[m, k] \times [k, n]$ 的 FLOPs 是 $2mkn$。

**矩阵乘 FLOPs 公式**：对于 $Y = XW$，其中 $X \in \mathbb{R}^{m \times k}$、$W \in \mathbb{R}^{k \times n}$，输出 $Y \in \mathbb{R}^{m \times n}$，每个输出元素需要 $k$ 次乘 + $k$ 次加 = $2k$ ops，总共 $2mkn$ ops。

把这个公式套到 Attention 和 FFN 的每个 matmul 上，逐项求 FLOPs。

In [7]:
# === Attention 的前向 FLOPs ===
# 每项 matmul: [m, k] × [k, n] → 2*m*k*n
# 这里 m = B*S, k 和 n 取决于具体投影

m = B * S   # token 总数

# Q 投影: [B*S, D] × [D, D] → 2*B*S*D*D
flops_q = 2 * m * D * (H * d_h)
# K 投影: [B*S, D] × [D, K*d_h] → 2*B*S*D*K*d_h
flops_k = 2 * m * D * (K * d_h)
# V 投影: 同 K
flops_v = 2 * m * D * (K * d_h)
# QK^T: [B, H, S, d_h] × [B, H, d_h, S] → 2*B*H*S*S*d_h = 2*B*S^2*D
flops_qkt = 2 * B * H * S * S * d_h
# AV:   [B, H, S, S] × [B, H, S, d_h] → 2*B*H*S*S*d_h = 2*B*S^2*D
flops_av = 2 * B * H * S * S * d_h
# O 投影: [B*S, D] × [D, D] → 2*B*S*D*D
flops_o = 2 * m * D * (H * d_h)

attn_flops = flops_q + flops_k + flops_v + flops_qkt + flops_av + flops_o
print(f"=== Attention 前向 FLOPs（MHA, B={B}, S={S}, D={D}） ===")
print(f"Q 投影:  {flops_q:,}")
print(f"K 投影:  {flops_k:,}  (GQA, K头={K})")
print(f"V 投影:  {flops_v:,}")
print(f"QK^T:    {flops_qkt:,}  ← S² 项")
print(f"AV:      {flops_av:,}  ← S² 项")
print(f"O 投影:  {flops_o:,}")
print(f"合计:    {attn_flops:,}  ≈ {attn_flops/1e9:.2f} GFLOPs")
print(f"\n公式 8BSD² + 4BS²D（MHA）= {8*B*S*D*D + 4*B*S*S*D:,} 误差很小")

=== Attention 前向 FLOPs（MHA, B=8, S=4096, D=576） ===
Q 投影:  21,743,271,936
K 投影:  7,247,757,312  (GQA, K头=3)
V 投影:  7,247,757,312
QK^T:    154,618,822,656  ← S² 项
AV:      154,618,822,656  ← S² 项
O 投影:  21,743,271,936
合计:    367,219,703,808  ≈ 367.22 GFLOPs

公式 8BSD² + 4BS²D（MHA）= 396,210,733,056 误差很小


In [8]:
# === FFN 的前向 FLOPs（SwiGLU: gate, up, down 三个 matmul） ===
flops_gate = 2 * m * D * FF
flops_up   = 2 * m * D * FF
flops_down = 2 * m * FF * D
ffn_flops = flops_gate + flops_up + flops_down

print(f"=== FFN 前向 FLOPs ===")
print(f"gate: {flops_gate:,}")
print(f"up:   {flops_up:,}")
print(f"down: {flops_down:,}")
print(f"合计: {ffn_flops:,}  (公式 6BSDF = {6*B*S*D*FF:,})")
print(f"代入 F=8D/3: 16BSD² = {16*B*S*D*D:,}  ≈ {ffn_flops/1e9:.2f} GFLOPs")

# === 单层 + 全局 ===
per_layer_flops = attn_flops + ffn_flops
total_flops = L * per_layer_flops + 2 * B * S * D * V  # 加 unembedding

# 闭式公式: 2BSD(12LD + 2LS + V)
closed_form = 2 * B * S * D * (12 * L * D + 2 * L * S + V)

print(f"\n=== 全局前向 FLOPs ===")
print(f"单层 (Attention+FFN): {per_layer_flops:,}  ≈ {per_layer_flops/1e9:.2f} GFLOPs")
print(f"unembedding [B*S,D]×[D,V]: {2*B*S*D*V:,}")
print(f"{L} 层 + unembedding: {total_flops:,}  ≈ {total_flops/1e12:.2f} TFLOPs")
print(f"\n闭式公式 2BSD(12LD + 2LS + V) = {closed_form:,}")
print(f"  误差: {abs(total_flops-closed_form)/total_flops*100:.2f}%")

=== FFN 前向 FLOPs ===
gate: 57,982,058,496
up:   57,982,058,496
down: 57,982,058,496
合计: 173,946,175,488  (公式 6BSDF = 173,946,175,488)
代入 F=8D/3: 16BSD² = 173,946,175,488  ≈ 173.95 GFLOPs

=== 全局前向 FLOPs ===
单层 (Attention+FFN): 541,165,879,296  ≈ 541.17 GFLOPs
unembedding [B*S,D]×[D,V]: 1,855,425,871,872
30 层 + unembedding: 18,090,402,250,752  ≈ 18.09 TFLOPs

闭式公式 2BSD(12LD + 2LS + V) = 18,960,133,128,192
  误差: 4.81%


## 4. 反向传播：FLOPs 约为前向的 2 倍

反向传播要为每个 op 计算两类梯度：对权重的梯度 $\partial L / \partial W$ 和对输入的梯度 $\partial L / \partial X$。每个梯度都是一次 matmul，所以反向的总 FLOPs 大约是前向的 2 倍。

工程上常说的「6 倍经验法则」就是从这里来：forward 1 份 + backward 2 份 = 3 份 FLOPs；考虑 step 间还有优化器更新等开销，再乘 2 留余量。

**估算单步训练 FLOPs**：$\text{FLOPs}_{\text{step}} \approx 6 \cdot 2BSD \cdot (12LD + 2LS + V) \approx 6PD$（当 $S$ 不大时，简化为参数量乘 token 数的 6 倍）。

In [9]:
# === 训练 FLOPs 经验法则：6 × P × tokens ===
# SmolLM2-135M 训练 2T tokens 的总 FLOPs
P_smolLM = 135e6
tokens_smolLM = 2e12
train_flops_smolLM = 6 * P_smolLM * tokens_smolLM

print(f"=== SmolLM2-135M 训练 FLOPs ===")
print(f"参数量 P = {P_smolLM/1e6:.0f}M")
print(f"训练 tokens = {tokens_smolLM/1e12:.0f}T")
print(f"总训练 FLOPs = 6PD = {train_flops_smolLM:.2e}")
print(f"  = {train_flops_smolLM/1e21:.2f} ZFLOPs")

# 7B 模型训练 1.5T tokens（LLaMA-1 配置）
P_7b = 7e9
tokens_7b = 1.5e12
train_flops_7b = 6 * P_7b * tokens_7b
print(f"\n=== 7B 模型训练 FLOPs ===")
print(f"参数量 P = {P_7b/1e9:.0f}B, tokens = {tokens_7b/1e12:.1f}T")
print(f"总训练 FLOPs = 6PD = {train_flops_7b:.2e} = {train_flops_7b/1e21:.2f} ZFLOPs")
print(f"在 2048 张 A100 (312 TFLOPs/s) 上理论耗时: {train_flops_7b/(2048*312e12)/86400:.1f} 天")

=== SmolLM2-135M 训练 FLOPs ===
参数量 P = 135M
训练 tokens = 2T
总训练 FLOPs = 6PD = 1.62e+21
  = 1.62 ZFLOPs

=== 7B 模型训练 FLOPs ===
参数量 P = 7B, tokens = 1.5T
总训练 FLOPs = 6PD = 6.30e+22 = 63.00 ZFLOPs
在 2048 张 A100 (312 TFLOPs/s) 上理论耗时: 1.1 天


## 5. 推理显存：weights + KV cache + peak activations

推理时显存分三块：

1. **权重**：$P$ 个参数，FP16 下 $2P$ bytes，INT8 下 $P$ bytes
2. **KV cache**：每个 token 在每层要存 K 和 V 两个张量
3. **峰值激活**：单层 forward 时的临时张量（前向结束后立即释放）

KV cache 的显存和 batch、序列长、层数、KV 头数都相关。每个 token 每层的 KV 大小是 $2 \cdot K \cdot d_h$（K 和 V 各一份），乘以 batch、序列长、层数、bytes/元素就是总 KV cache 显存。

$$\text{KV cache} = B \cdot S \cdot K \cdot d_h \cdot L \cdot 2 \cdot \text{bytes/element}$$

In [10]:
# === KV cache 显存手算（7B 模型） ===
# 7B 模型典型配置
D_7b = 4096
L_7b = 32
H_7b = 32
K_7b = 32        # LLaMA-1 是 MHA, K=H
d_h_7b = D_7b // H_7b   # 128
V_7b = 32000

# 估算参数量
P_7b_formula = 2 * V_7b * D_7b + 12 * L_7b * D_7b * D_7b
print(f"=== 7B 模型 ===")
print(f"公式 P ≈ 2VD + 12LD² = {P_7b_formula/1e9:.2f}B")

# FP16 权重显存
weights_fp16 = 2 * P_7b_formula
print(f"权重 FP16: {weights_fp16/1e9:.2f} GB")

# KV cache: 不同 batch 和 seq 组合
scenarios = [
    ("单用户 S=2048",  1,    2048),
    ("单用户 S=8192",  1,    8192),
    ("B=8, S=2048",    8,    2048),
    ("B=32, S=2048",   32,   2048),
    ("B=32, S=8192",   32,   8192),
]

print(f"\n=== KV cache 显存（FP16, K=V 头数={K_7b}, head_dim={d_h_7b}） ===")
print(f"{'场景':<20} {'KV cache (GB)':>15}")
for name, b, s in scenarios:
    kv_elems = b * s * K_7b * d_h_7b * L_7b * 2  # ×2 for K and V
    kv_bytes = kv_elems * 2   # FP16
    print(f"{name:<20} {kv_bytes/1e9:>15.2f}")

=== 7B 模型 ===
公式 P ≈ 2VD + 12LD² = 6.70B
权重 FP16: 13.41 GB

=== KV cache 显存（FP16, K=V 头数=32, head_dim=128） ===
场景                     KV cache (GB)
单用户 S=2048                      1.07
单用户 S=8192                      4.29
B=8, S=2048                     8.59
B=32, S=2048                   34.36
B=32, S=8192                  137.44


In [11]:
# === 推理显存的三种主导情形 ===
# 横轴: batch size；比较 weights, KV cache, activations

import numpy as np

S_fixed = 2048
B_range = np.array([1, 2, 4, 8, 16, 32, 64, 128])

weights_gb = np.full_like(B_range, 2 * P_7b_formula / 1e9, dtype=float)
kv_gb = B_range * S_fixed * K_7b * d_h_7b * L_7b * 2 * 2 / 1e9
# 峰值激活（粗略: 单层 BSD × ~10 + BNS²）
act_gb = (10 * B_range * S_fixed * D_7b + B_range * H_7b * S_fixed**2) * 2 / 1e9

print(f"{'B':>5} {'weights':>10} {'KV cache':>10} {'activations':>12} {'dominant':>12}")
print("-" * 55)
for i, b in enumerate(B_range):
    parts = {'weights': weights_gb[i], 'KV': kv_gb[i], 'act': act_gb[i]}
    dom = max(parts, key=parts.get)
    print(f"{b:>5} {weights_gb[i]:>9.2f}G {kv_gb[i]:>9.2f}G {act_gb[i]:>11.2f}G {dom:>12}")

print("\n=== 三种主导情形 ===")
print("(1) 小 batch + 短序列：weights 占主导，KV/激活都很小")
print("(2) 长 S：激活里 S² 项主导，KV cache 也线性涨")
print("(3) 大 batch：KV cache 线性涨，激活里的 BSD 和 BHS² 都涨")

    B    weights   KV cache  activations     dominant
-------------------------------------------------------
    1     13.41G      1.07G        0.44G      weights
    2     13.41G      2.15G        0.87G      weights
    4     13.41G      4.29G        1.74G      weights
    8     13.41G      8.59G        3.49G      weights
   16     13.41G     17.18G        6.98G           KV
   32     13.41G     34.36G       13.96G           KV
   64     13.41G     68.72G       27.92G           KV
  128     13.41G    137.44G       55.83G           KV

=== 三种主导情形 ===
(1) 小 batch + 短序列：weights 占主导，KV/激活都很小
(2) 长 S：激活里 S² 项主导，KV cache 也线性涨
(3) 大 batch：KV cache 线性涨，激活里的 BSD 和 BHS² 都涨


## 6. 训练显存：16P + activations 的来源

训练显存比推理复杂得多。AdamW + 混合精度下，每个参数要配套存这些东西：

| 项目 | bytes/参数 | 说明 |
|:---|:---|:---|
| FP32 master 权重 | 4P | 混合精度下的主权重 |
| BF16 transient 权重 | 2P | 前向时用的低精度副本 |
| AdamW 一阶矩 $m$ | 4P | FP32 累积 |
| AdamW 二阶矩 $v$ | 4P | FP32 累积 |
| FP32 梯度 | 4P | 反向时 BF16 算，FP32 累积 |
| 小计 | **16P** | 这就是「固定开销」 |

加上激活就是完整训练显存。激活部分取决于 $B, S, D, L$，公式为 $L \cdot (14BSD + BNS^2)$，可以用 gradient checkpointing 大幅压缩。

In [12]:
# === 7B 模型训练显存手算 ===
P = P_7b_formula

# 固定开销 16P
fp32_master  = 4 * P
bf16_weights = 2 * P
adam_m       = 4 * P
adam_v       = 4 * P
fp32_grad    = 4 * P    # 反向时 BF16 计算，FP32 累积
fixed = fp32_master + bf16_weights + adam_m + adam_v + fp32_grad

print(f"=== 7B 训练显存固定开销 ===")
print(f"FP32 master 权重: {fp32_master/1e9:>6.2f} GB")
print(f"BF16 权重副本:    {bf16_weights/1e9:>6.2f} GB")
print(f"AdamW m:          {adam_m/1e9:>6.2f} GB")
print(f"AdamW v:          {adam_v/1e9:>6.2f} GB")
print(f"FP32 梯度:        {fp32_grad/1e9:>6.2f} GB")
print(f"固定小计:         {fixed/1e9:>6.2f} GB  (= 16P)")

# 激活: L*(14BSD + BHS²) (BF16 = 2 bytes)
B_train, S_train = 8, 4096
D, H, L = D_7b, H_7b, L_7b
per_layer_act = 14 * B_train * S_train * D + B_train * H * S_train * S_train
act_total = L * per_layer_act * 2  # BF16

print(f"\n=== 激活显存（B={B_train}, S={S_train}, BF16） ===")
print(f"单层 14BSD+BHS²: {per_layer_act:,} elements")
print(f"L={L} 层合计:    {L*per_layer_act:,} elements = {act_total/1e9:.2f} GB")

print(f"\n=== 训练总显存 ===")
total_train = fixed + act_total
print(f"固定 16P:        {fixed/1e9:>6.2f} GB ({fixed/total_train*100:.0f}%)")
print(f"激活:            {act_total/1e9:>6.2f} GB ({act_total/total_train*100:.0f}%)")
print(f"合计:            {total_train/1e9:>6.2f} GB")
print(f"单张 A100 80GB: {'装不下' if total_train/1e9 > 80 else '放得下'}")

=== 7B 训练显存固定开销 ===
FP32 master 权重:  26.82 GB
BF16 权重副本:     13.41 GB
AdamW m:           26.82 GB
AdamW v:           26.82 GB
FP32 梯度:         26.82 GB
固定小计:         120.68 GB  (= 16P)

=== 激活显存（B=8, S=4096, BF16） ===
单层 14BSD+BHS²: 6,174,015,488 elements
L=32 层合计:    197,568,495,616 elements = 395.14 GB

=== 训练总显存 ===
固定 16P:        120.68 GB (23%)
激活:            395.14 GB (77%)
合计:            515.82 GB
单张 A100 80GB: 装不下


## 7. Gradient Checkpointing：用算力换显存

激活在长序列下暴涨，单卡装不下。Gradient checkpointing 的思路是：前向时不保留每层的中间激活，只保留少数几个「检查点」层的输入；反向时从检查点开始重算前向，临时得到需要的激活。

代价是前向多算一次，显存从 $O(L)$ 降到 $O(\sqrt{L})$。具体来说，把 $L$ 层分成 $\sqrt{L}$ 段，每段 $\sqrt{L}$ 层，只存每段的边界输入——存储 $\sqrt{L}$ 个边界，反向时每段重算 $\sqrt{L}$ 层，总开销 $O(\sqrt{L})$ 存储 + $O(L)$ 额外计算。

PyTorch 提供了 `torch.utils.checkpoint.checkpoint` API。

In [13]:
# === Gradient checkpointing 实测对比（CPU 上模拟） ===
import torch.utils.checkpoint as cp

class CheckpointedBlock(nn.Module):
    """包装一层 TransformerBlock，前向时使用 gradient checkpointing"""
    def __init__(self, block):
        super().__init__()
        self.block = block
    def forward(self, x):
        # 要求输入 requires_grad=True 才能重算
        return cp.checkpoint(self.block, x, use_reentrant=False)

# 构造两个等价的 MiniLM：一个不 checkpoint，一个每层都 checkpoint
torch.manual_seed(0)
model_plain = MiniLM(V, D_7b // 4, 8, 8, 8, D_7b // 4 * 8 // 3)
# 缩小规模便于 CPU 测试：D=1024, L=8
model_plain = MiniLM(V=1000, D=256, L=8, H=4, K=4, FF=512)

torch.manual_seed(0)
model_ckpt = MiniLM(V=1000, D=256, L=8, H=4, K=4, FF=512)
for i, blk in enumerate(model_ckpt.blocks):
    model_ckpt.blocks[i] = CheckpointedBlock(blk)

# 测量显存（CPU 上用 allocated 不可用，用 torch.cuda 测需 GPU；这里只测能跑通）
idx = torch.randint(0, 1000, (2, 64))
logits_plain = model_plain(idx)
logits_ckpt = model_ckpt(idx)
print(f"模型 A（plain）输出 shape: {list(logits_plain.shape)}")
print(f"模型 B（checkpoint）输出 shape: {list(logits_ckpt.shape)}")
print(f"两者权重相同: {torch.allclose(model_plain.embed.weight, model_ckpt.embed.weight)}")

# 用公式估算两种方案的激活显存
L_test = 8
B_test, S_test, D_test = 2, 64, 256
H_test = 4
per_layer_act_test = 14 * B_test * S_test * D_test + B_test * H_test * S_test * S_test

# Plain: 存所有 L 层
plain_act = L_test * per_layer_act_test
# Checkpoint (sqrt 策略): 只存 sqrt(L) 个边界，反向时重算
sqrt_L = int(L_test ** 0.5)
ckpt_act = sqrt_L * per_layer_act_test  # 粗略：存储降为 sqrt(L)/L

print(f"\n=== 激活显存对比（L={L_test}, B={B_test}, S={S_test}） ===")
print(f"Plain:            L · per-layer = {plain_act:,} elements")
print(f"Checkpoint (√L):   √L · per-layer = {ckpt_act:,} elements")
print(f"节省比例:          {1 - ckpt_act/plain_act:.0%}")
print(f"代价: 前向多算一次 (≈ +33% FLOPs)")

模型 A（plain）输出 shape: [2, 64, 1000]
模型 B（checkpoint）输出 shape: [2, 64, 1000]
两者权重相同: False

=== 激活显存对比（L=8, B=2, S=64） ===
Plain:            L · per-layer = 3,932,160 elements
Checkpoint (√L):   √L · per-layer = 983,040 elements
节省比例:          75%
代价: 前向多算一次 (≈ +33% FLOPs)


## 8. MoE 场景：总参数 vs 激活参数

MoE（Mixture of Experts）模型的 FFN 被替换成了 N 个并行的专家 FFN，每个 token 通过一个 router 选 top-k 个专家。这改变了参数和 FLOPs 的算法。

**总参数量** vs **激活参数量**：

- **总参数量**：所有专家的 FFN 参数都算上。Mixtral 8x7B 是 8 个专家，每个专家 FFN 大小和 dense 7B 一样，加上 attention/embedding，总参数约 46.7B。
- **激活参数量**：每个 token 只激活 top-2 个专家，所以 forward 时实际「活跃」的参数只有 attention + 2 个专家 FFN，约 13B。

**FLOPs** 要按激活参数算，因为只有 top-k 专家的 FFN 真的做了 matmul。

**KV cache 不变**：MoE 只把 FFN 换成专家，attention 仍然是 dense 的共享结构，所以 KV cache 公式不变。

In [14]:
# === Mixtral 8x7B 参数量手算 ===
# 假设每个专家 ≈ LLaMA-7B 的 FFN 配置
D_mix = 4096
L_mix = 32
H_mix = 32
K_mix = 8        # GQA in Mixtral
d_h_mix = D_mix // H_mix
FF_mix = 14336   # Mixtral FFN dim
V_mix = 32000
E = 8            # 8 个专家
top_k = 2

# 每层 attention（dense 部分）
attn_per_layer = 4 * D_mix * D_mix  # MHA 近似

# 每层 FFN: E 个专家，每个 3*D*FF
ffn_all_experts = E * 3 * D_mix * FF_mix
ffn_active = top_k * 3 * D_mix * FF_mix   # top-k 激活

per_layer_total = attn_per_layer + ffn_all_experts
per_layer_active = attn_per_layer + ffn_active

P_total = V_mix * D_mix + L_mix * per_layer_total + D_mix   # tie=False 时 unembed 单算
P_active = V_mix * D_mix + L_mix * per_layer_active + D_mix

print(f"=== Mixtral 8x7B ===")
print(f"专家数 E = {E}, top-k = {top_k}")
print(f"每层 Attention:    {attn_per_layer/1e6:.0f}M")
print(f"每层 FFN (全部 {E} 专家): {ffn_all_experts/1e6:.0f}M")
print(f"每层 FFN (激活 {top_k} 个): {ffn_active/1e6:.0f}M")
print(f"\n总参数量 P_total:   {P_total/1e9:.1f}B  (论文报告 46.7B)")
print(f"激活参数量 P_active: {P_active/1e9:.1f}B  (论文报告 ~13B)")
print(f"激活比 = {P_active/P_total:.1%}")

# FLOPs 用 P_active 算
tokens = 1e12
flops_moe = 6 * P_active * tokens
flops_dense_equiv = 6 * P_total * tokens
print(f"\n训练 {tokens/1e12:.0f}T tokens:")
print(f"  MoE 实际 FLOPs (6·P_active·D): {flops_moe/1e21:.2f} ZFLOPs")
print(f"  等效 dense FLOPs (6·P_total·D): {flops_dense_equiv/1e21:.2f} ZFLOPs")
print(f"  MoE 用 1/{flops_dense_equiv/flops_moe:.1f} 的算力训出更大模型")

=== Mixtral 8x7B ===
专家数 E = 8, top-k = 2
每层 Attention:    67M
每层 FFN (全部 8 专家): 1409M
每层 FFN (激活 2 个): 352M

总参数量 P_total:   47.4B  (论文报告 46.7B)
激活参数量 P_active: 13.6B  (论文报告 ~13B)
激活比 = 28.6%

训练 1T tokens:
  MoE 实际 FLOPs (6·P_active·D): 81.32 ZFLOPs
  等效 dense FLOPs (6·P_total·D): 284.25 ZFLOPs
  MoE 用 1/3.5 的算力训出更大模型


In [15]:
# === MoE 下 KV cache 不变 ===
# 因为 attention 仍然是 dense 共享的，专家只替换 FFN
B_infer, S_infer = 1, 8192

# Mixtral 的 KV cache（和 dense 7B 一样的公式）
kv_mixtral = B_infer * S_infer * K_mix * d_h_mix * L_mix * 2 * 2  # ×2 K/V, ×2 FP16
print(f"=== KV cache 对比（B={B_infer}, S={S_infer}, FP16） ===")
print(f"Mixtral 8x7B: {kv_mixtral/1e9:.2f} GB")
print(f"dense 7B:     {B_infer * S_infer * K_7b * d_h_7b * L_7b * 2 * 2 / 1e9:.2f} GB")
print(f"两者公式完全相同：B·S·K·d_h·L·2·2")
print(f"\n关键观察：MoE 增大总参数量但推理 FLOPs/KV cache 都按 active 算")
print(f"  这是 MoE 相对 dense 大模型的核心优势：单位算力下质量更高")

=== KV cache 对比（B=1, S=8192, FP16） ===
Mixtral 8x7B: 1.07 GB
dense 7B:     4.29 GB
两者公式完全相同：B·S·K·d_h·L·2·2

关键观察：MoE 增大总参数量但推理 FLOPs/KV cache 都按 active 算
  这是 MoE 相对 dense 大模型的核心优势：单位算力下质量更高


## 小结

确认你已经搞懂了这些公式：

- [ ] 单层 Attention 参数 ≈ $4D^2$（MHA），FFN（SwiGLU）≈ $3DF$，代入 $F=8D/3$ 后 FFN ≈ $8D^2$
- [ ] 全局参数量公式：$P \approx 2VD + 12LD^2$
- [ ] 单层激活 ≈ $14BSD + BNS^2$，平方项来自 $S \times S$ attention matrix
- [ ] 单层前向 FLOPs ≈ $8BSD^2 + 4BS^2D + 16BSD^2$，全局 ≈ $2BSD(12LD + 2LS + V)$
- [ ] 反向 FLOPs ≈ 2 × forward，训练经验法则 6PD
- [ ] KV cache 大小：$B \cdot S \cdot K \cdot d_h \cdot L \cdot 2 \cdot \text{bytes/element}$
- [ ] 训练显存固定开销 = 16P（FP32 master + BF16 transient + AdamW m/v + FP32 grad）
- [ ] Gradient checkpointing 把激活从 $O(L)$ 降到 $O(\sqrt{L})$，代价是前向重算
- [ ] MoE 总参数按全部专家算，激活参数/FLOPs/KV cache 按 top-k 激活的专家算

## 作业

> 可以用 AI 帮忙解释思路，但不建议直接让 AI「做完这道题」。

**作业 1：估算 LLaMA-2 70B 的参数量和 FP16 权重显存**

LLaMA-2 70B 配置：$D=8192$，$L=80$，$H=64$，$K=8$（GQA），$V=32000$。用公式 $P \approx 2VD + 12LD^2$ 算参数量，再算 FP16 下的权重显存。

小提示：参数量 ≈ $2VD + 12LD^2$；FP16 下每个参数 2 bytes。

In [16]:
# 作业 1：LLaMA-2 70B 参数量和权重显存
D_l2 = 8192
L_l2 = 80
V_l2 = 32000

# TODO: 用公式估算参数量
P_l2 = 2 * V_l2 * D_l2 + 12 * L_l2 * D_l2 ** 2

# TODO: 算 FP16 权重显存（bytes）
weights_bytes_l2 = 2 * P_l2

# 验证
assert P_l2 is not None, "请先用公式计算 P_l2"
assert weights_bytes_l2 is not None, "请先计算 weights_bytes_l2"
expected_P = 2 * V_l2 * D_l2 + 12 * L_l2 * D_l2 ** 2
assert abs(P_l2 - expected_P) / expected_P < 0.01, f"参数量应约 {expected_P/1e9:.1f}B"
expected_bytes = 2 * expected_P
assert weights_bytes_l2 == expected_bytes, f"显存应为 {expected_bytes/1e9:.1f} GB"
print(f"作业 1 通过：")
print(f"  LLaMA-2 70B 参数量 ≈ {P_l2/1e9:.1f}B")
print(f"  FP16 权重显存 ≈ {weights_bytes_l2/1e9:.1f} GB")

作业 1 通过：
  LLaMA-2 70B 参数量 ≈ 64.9B
  FP16 权重显存 ≈ 129.9 GB


**作业 2：算 13B 模型 batch=8 seq=4096 的训练显存**

13B 模型（$D=5120$, $L=40$, $H=40$, MHA），AdamW + BF16 混合精度训练。batch=8, seq=4096。算：固定开销（16P）、激活（BF16）、总显存。

小提示：固定 = 16P bytes；激活 = $L \cdot (14BSD + BHS^2)$ 个元素 × 2 bytes。

In [17]:
# 作业 2：13B 模型训练显存
P_13b = 13e9     # 简化：用 13B 这个数字
B_ex, S_ex = 8, 4096
D_ex, H_ex, L_ex = 5120, 40, 40

# TODO: 固定开销（bytes）
fixed_bytes = 16 * P_13b

# TODO: 激活（bytes，BF16 = 2 bytes/元素）
act_elements = L_ex * (14 * B_ex * S_ex * D_ex + B_ex * H_ex * S_ex * S_ex)
act_bytes = act_elements * 2

# TODO: 总显存（GB）
total_gb = (fixed_bytes + act_bytes) / 1e9

assert fixed_bytes is not None and act_elements is not None and total_gb is not None
assert fixed_bytes == 16 * P_13b, "固定开销应为 16P"
assert act_elements == L_ex * (14 * B_ex * S_ex * D_ex + B_ex * H_ex * S_ex * S_ex)
print(f"作业 2 通过：")
print(f"  固定开销: {fixed_bytes/1e9:.1f} GB")
print(f"  激活:     {act_bytes/1e9:.1f} GB")
print(f"  总显存:   {total_gb:.1f} GB")
print(f"  需要 {int((total_gb+79)//80)} 张 A100 80GB")

作业 2 通过：
  固定开销: 208.0 GB
  激活:     617.4 GB
  总显存:   825.4 GB
  需要 11 张 A100 80GB


**作业 3：估算 Mixtral 8x7B 单次 prefill 的 FLOPs**

Mixtral 8x7B，top-2 路由，输入 $B=1$, $S=4096$。用激活参数量 $P_{\text{active}} \approx 13B$ 估算单次 prefill 的 forward FLOPs。

小提示：forward FLOPs ≈ $2 \cdot P_{\text{active}} \cdot B \cdot S$（当 unembedding 主导或简化估算时，可写作 $2BSD(12LD+2LS+V) \approx 6PD \cdot \text{tokens}$ 的 1/3，即 forward ≈ $2P_{\text{active}} \cdot$ tokens）。

In [18]:
# 作业 3：Mixtral 8x7B prefill FLOPs
P_active_mix = 13e9
B_ex3, S_ex3 = 1, 4096
tokens_ex3 = B_ex3 * S_ex3

# TODO: forward FLOPs（简化估算：2 * P_active * tokens）
forward_flops = 2 * P_active_mix * tokens_ex3

assert forward_flops is not None
expected = 2 * P_active_mix * tokens_ex3
assert forward_flops == expected, f"应等于 {expected}"
print(f"作业 3 通过：")
print(f"  tokens = {tokens_ex3}")
print(f"  forward FLOPs ≈ {forward_flops:.2e} = {forward_flops/1e12:.2f} TFLOPs")
print(f"  注意：这是激活参数版本的估算，比 dense 46.7B 小很多")

作业 3 通过：
  tokens = 4096
  forward FLOPs ≈ 1.06e+14 = 106.50 TFLOPs
  注意：这是激活参数版本的估算，比 dense 46.7B 小很多


## 参考资料

- Pope et al., [Efficient Memory Management for Large Language Model Serving with PagedAttention](https://arxiv.org/abs/2309.06180), 2023 — KV cache 显存分析的经典出处
- Korthikanti et al., [Reducing Activation Recomputation in Large Transformer Models](https://arxiv.org/abs/2205.05198), 2022 — Megatron-LM 的激活公式 $14BSD + BNS^2$ 来源
- Chen et al., [Training Deep Nets with Sublinear Memory Cost](https://arxiv.org/abs/1604.06174), 2016 — gradient checkpointing 原始论文
- Hoffmann et al., [Training Compute-Optimal Large Language Models](https://arxiv.org/abs/2203.15556), 2022 — Chinchilla scaling law，6PD 的来源
- Fedus et al., [Switch Transformers: Scaling to Trillion Parameter Models](https://arxiv.org/abs/2101.03961), 2021 — MoE 的参数 vs 激活参数分析